# Experiment No. 09 - Personalized AI Assistant using RAG and LoRA

Run cells from top to bottom.

In [ ]:
!pip -q install transformers sentence-transformers faiss-cpu accelerate peft datasets

## Aim
Build a personalized assistant that retrieves information from personal documents. Includes a small optional LoRA fine-tuning demonstration.

In [ ]:
import torch, numpy as np, faiss
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
docs=[
 "I am an AIML student interested in machine learning and software development.",
 "FlowGuard monitors water flow and detects leaks using IoT sensors and an ESP32.",
 "Trust Index is a framework for evaluating trust and hallucinations in RAG systems."
]
embedder=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
vectors=embedder.encode(docs).astype("float32")
index=faiss.IndexFlatL2(vectors.shape[1]); index.add(vectors)
tokenizer=AutoTokenizer.from_pretrained("google/flan-t5-small")
model=AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
device="cuda" if torch.cuda.is_available() else "cpu"; model=model.to(device)

In [ ]:
def ask(question):
 q=embedder.encode([question]).astype("float32")
 _,ids=index.search(q,2)
 context=" ".join(docs[i] for i in ids[0])
 prompt=f"Answer using only this context. If unknown, say so. Context: {context} Question: {question}"
 inputs=tokenizer(prompt,return_tensors="pt",truncation=True).to(device)
 result=model.generate(**inputs,max_new_tokens=100)
 return tokenizer.decode(result[0],skip_special_tokens=True)
q=input("Ask a question about your documents: ")
print(ask(q))

## Optional LoRA fine-tuning
This is a minimal adapter setup. For actual training, prepare a larger domain-specific dataset and use a Trainer or TRL training loop.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
base=AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")
config=LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM,r=8,lora_alpha=16,lora_dropout=0.1,target_modules=["q","v"])
lora_model=get_peft_model(base,config)
lora_model.print_trainable_parameters()
print("LoRA adapter configured. Fine-tune it on a prepared question-answer dataset before evaluating response quality.")

## Result
The RAG assistant retrieves relevant document text and generates a grounded answer. The LoRA cell configures parameter-efficient adapters; it does not claim a fine-tuning result without a training dataset.